# CSPF — audit and upgrade notebook (Colab / Kaggle, no API needed)

**Purpose.** (1) find out why original-text accuracy is below chance on several datasets while perturbed-text accuracy is high (prediction-alignment audit); (2) re-run all eight models with identical preprocessing and a collision-free key; (3) recompute every table of the paper (pair-weighted metrics, cluster bootstrap, paired tests, within-family severity/position effects, Holm/BH); (4) replace the degenerate fidelity score by an NLI-based one and check it against human labels; (5) add domain-matched control models; (6) export all tables as CSV + `numbers.json` to paste into the manuscript.

**You need:** the released corpus CSV (GitHub `hadis-1/CSPF`) with at least the columns listed in the config cell, and (optional) your own fine-tuned checkpoints on Drive. Runtime: GPU (T4 is enough). Steps are resumable: predictions are cached per model.


In [ ]:
!pip -q install transformers sentence-transformers vaderSentiment scikit-learn statsmodels pandas pyarrow scipy tqdm
import os, json, re, math, itertools, warnings, numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
OUT='cspf_out'; os.makedirs(OUT,exist_ok=True)
print(torch.cuda.is_available())

## 0. Configuration

In [ ]:
# Option A (no upload needed): clone your GitHub repo inside Colab and locate the corpus file
!git clone -q https://github.com/hadis-1/CSPF /content/CSPF
!find /content/CSPF -type f \( -name "*.csv" -o -name "*.parquet" -o -name "*.zip" \) -size +1M | head


In [ ]:
CORPUS_CSV = '/content/cspf_corpus.csv'      # <- path to the released corpus (upload to Colab or add as Kaggle dataset)
# map YOUR column names to the canonical ones used below
COLS = dict(dataset='source_dataset', idx='review_idx', orig='original_review', pert='perturbed_review',
            label='sentiment', family='perturbation_family', severity='severity', position='position_strategy')
MAXLEN = 128; BATCH = 128
# Off-the-shelf models (no API). Replace/add your own fine-tuned checkpoints, e.g. '/content/drive/MyDrive/ckpt/bert'.
MODELS = {
 'distilbert-sst2':'distilbert-base-uncased-finetuned-sst-2-english',
 'bert-sst2':'textattack/bert-base-uncased-SST-2',
 'roberta-sst2':'textattack/roberta-base-SST-2',
 'albert-sst2':'textattack/albert-base-v2-SST-2',
 'roberta-large-sentiment':'siebert/sentiment-roberta-large-english',
 # domain-matched controls (trained on tweets)
 'twitter-roberta':'cardiffnlp/twitter-roberta-base-sentiment-latest',
 'bertweet':'finiteautomata/bertweet-base-sentiment-analysis',
}
SUBSAMPLE_PER_DATASET = None   # e.g. 3000 to debug quickly; None = full corpus
SEED=42

## 1. Load corpus and audit keys / labels / preprocessing

In [ ]:
df = pd.read_csv(CORPUS_CSV)
df = df.rename(columns={v:k for k,v in COLS.items()})
need=list(COLS); assert all(c in df.columns for c in need), [c for c in need if c not in df.columns]
df['key'] = df.dataset.astype(str)+'::'+df.idx.astype(str)          # composite key (dataset, review_idx)
print(len(df),'pairs;',df.key.nunique(),'composite keys;',df.idx.nunique(),'raw review_idx values')
# 1a. is review_idx unique across datasets? (collision test)
coll = df.groupby('idx').dataset.nunique()
print('review_idx shared by >1 dataset:', int((coll>1).sum()),'of',len(coll))
# 1b. one original text per key?
multi = df.groupby('key').orig.nunique(); print('keys with >1 distinct original text:', int((multi>1).sum()))
# 1c. labels
print(df.label.value_counts(dropna=False).to_dict())
df['y'] = df.label.map(lambda s: 1 if str(s).lower() in ('positive','pos','1','+1') else (0 if str(s).lower() in ('negative','neg','0','-1') else np.nan))
assert df.y.notna().all(), 'non-binary labels present: drop or map them'
# 1d. text length per dataset (reported in the paper's dataset table)
df['nwords']=df.orig.str.split().str.len()
print(df.drop_duplicates('key').groupby('dataset').nwords.agg(['count','median','mean']).round(1))
if SUBSAMPLE_PER_DATASET:
    keep=df.drop_duplicates('key').groupby('dataset',group_keys=False).apply(lambda g:g.sample(min(len(g),SUBSAMPLE_PER_DATASET),random_state=SEED)).key
    df=df[df.key.isin(set(keep))].reset_index(drop=True)

## 2. Prediction (identical preprocessing for original and perturbed; cached per model)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
def clean(t):               # SAME function for originals and perturbed texts
    t=str(t); t=re.sub(r'https?://\S+','http',t); t=re.sub(r'@\w+','@user',t); return t.strip()
def label_map(model):
    id2=getattr(model.config,'id2label',{}) or {}
    pos=[i for i,l in id2.items() if re.search(r'pos|label_1|^1$',str(l).lower())]
    neg=[i for i,l in id2.items() if re.search(r'neg|label_0|^0$',str(l).lower())]
    assert pos and neg, f'cannot map labels: {id2}'
    return int(pos[0]), int(neg[0])
@torch.no_grad()
def predict(texts, tok, model, pos, neg, dev):
    order=np.argsort([len(t) for t in texts]); out=np.zeros(len(texts)); conf=np.zeros(len(texts))
    for i in tqdm(range(0,len(texts),BATCH),leave=False):
        ids=order[i:i+BATCH]; enc=tok([texts[j] for j in ids],padding=True,truncation=True,max_length=MAXLEN,return_tensors='pt').to(dev)
        with torch.autocast('cuda',dtype=torch.float16,enabled=dev=='cuda'):
            lg=model(**enc).logits.float()
        two=torch.stack([lg[:,neg],lg[:,pos]],1).softmax(-1)      # ignore a neutral logit if present
        out[ids]=two[:,1].cpu().numpy(); conf[ids]=two.max(1).values.cpu().numpy()
    return out, conf
dev='cuda' if torch.cuda.is_available() else 'cpu'
uo = df.drop_duplicates('key')[['key','orig']].reset_index(drop=True)
uo['otxt']=uo.orig.map(clean); df['ptxt']=df.pert.map(clean)
preds={}
for name,hf in MODELS.items():
    f_o=f'{OUT}/pred_orig_{name}.parquet'; f_p=f'{OUT}/pred_pert_{name}.parquet'
    if os.path.exists(f_o) and os.path.exists(f_p): continue
    try:
        tok=AutoTokenizer.from_pretrained(hf); model=AutoModelForSequenceClassification.from_pretrained(hf).to(dev).eval()
        pos,neg=label_map(model)
    except Exception as e:
        print('SKIP',name,e); continue
    po,co=predict(uo.otxt.tolist(),tok,model,pos,neg,dev); pp,cp=predict(df.ptxt.tolist(),tok,model,pos,neg,dev)
    pd.DataFrame({'key':uo.key,'p_orig':po,'c_orig':co}).to_parquet(f_o); pd.DataFrame({'p_pert':pp,'c_pert':cp}).to_parquet(f_p)
    del model; torch.cuda.empty_cache(); print('done',name)

## 3. Alignment audit and corrected headline metrics

In [ ]:
rows=[]
for name in MODELS:
    f_o=f'{OUT}/pred_orig_{name}.parquet'
    if not os.path.exists(f_o): continue
    o=pd.read_parquet(f_o).set_index('key'); p=pd.read_parquet(f'{OUT}/pred_pert_{name}.parquet')
    d=df[['key','y','dataset','family','severity','position','idx']].copy()
    d['po']=(o.p_orig.reindex(d.key).values>0.5).astype(int); d['pp']=(p.p_pert.values>0.5).astype(int)
    d['co']=o.c_orig.reindex(d.key).values; d['cp']=p.c_pert.values
    d['flip']=(d.po!=d.pp).astype(int); d['ok_o']=(d.po==d.y).astype(int); d['ok_p']=(d.pp==d.y).astype(int)
    d.to_parquet(f'{OUT}/long_{name}.parquet')
    N=len(d); rows.append(dict(model=name,RS=d.flip.mean(),FRc=d[d.ok_o==1].flip.mean(),FRw=d[d.ok_o==0].flip.mean(),
        DR=((d.ok_o==1)&(d.ok_p==0)).mean(),IRR=((d.ok_o==0)&(d.ok_p==1)).mean(),acc_o=d.ok_o.mean(),acc_p=d.ok_p.mean()))
T2=pd.DataFrame(rows).round(4); T2['net_gain']=(T2.acc_p-T2.acc_o).round(4); print(T2.to_string()); T2.to_csv(f'{OUT}/Table2_model_level.csv',index=False)
# SANITY GATE: for meaning-preserving edits |acc_p-acc_o| should be small and accuracy should be clearly above 0.5 on every dataset
assert T2.net_gain.abs().max()<0.05, 'STOP: original/perturbed accuracy still disagree -> keep debugging (see per-dataset table)'

In [ ]:
# per-dataset audit (replaces Appendix Table A3)
aud=[]
for name in MODELS:
    f=f'{OUT}/long_{name}.parquet'
    if not os.path.exists(f): continue
    d=pd.read_parquet(f); g=d.groupby('dataset').agg(acc_o=('ok_o','mean'),acc_p=('ok_p','mean'),FR=('flip','mean'),n=('flip','size')); g['model']=name; aud.append(g.reset_index())
A3=pd.concat(aud).groupby('dataset')[['acc_o','acc_p','FR','n']].mean().round(3); A3['gain']=(A3.acc_p-A3.acc_o).round(3)
print(A3.sort_values('acc_o')); A3.to_csv(f'{OUT}/TableA3_dataset_audit.csv')
print('corr(DVI=FR, acc_o) =',np.corrcoef(A3.FR,A3.acc_o)[0,1].round(3))

**If the sanity gate fails**, test the three suspects in this order: (i) key collision — compare `df.idx.nunique()` with `df.key.nunique()` printed in section 1 (your first-pass join used `review_idx` alone); (ii) preprocessing — the first pipeline may have fed raw tweets to the original and cleaned text to the perturbed one (or vice versa): re-run section 2 with `clean` replaced by the identity for both and compare; (iii) labels — check for neutral/other labels collapsed into 'negative'. Report which one was the cause in the paper's Section 4.3.

## 4. Inference that respects dependence (cluster bootstrap, paired tests, multiplicity)

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from scipy import stats
rng=np.random.default_rng(SEED)
def cluster_boot(d, col='flip', B=1000):
    g=d.groupby('key')[col].agg(['sum','count']); s=g['sum'].values; c=g['count'].values; n=len(g); out=[]
    for _ in range(B):
        i=rng.integers(0,n,n); out.append(s[i].sum()/c[i].sum())
    return np.percentile(out,[2.5,97.5])
ci=[]
for name in MODELS:
    f=f'{OUT}/long_{name}.parquet'
    if os.path.exists(f):
        d=pd.read_parquet(f); lo,hi=cluster_boot(d); ci.append(dict(model=name,RS=d.flip.mean(),CI_lo=lo,CI_hi=hi))
CI=pd.DataFrame(ci).round(4); print(CI); CI.to_csv(f'{OUT}/Table2_cluster_CI.csv',index=False)
# Cochran's Q over models (same pairs), then pairwise McNemar with Holm
names=[n for n in MODELS if os.path.exists(f'{OUT}/long_{n}.parquet')]
F=np.column_stack([pd.read_parquet(f'{OUT}/long_{n}.parquet').flip.values for n in names]); k=F.shape[1]
Tj=F.sum(0); Li=F.sum(1); Q=(k-1)*(k*(Tj**2).sum()-Tj.sum()**2)/(k*Tj.sum()-(Li**2).sum()); print('Cochran Q =',Q,'p =',stats.chi2.sf(Q,k-1))
pr=[]
for a,b in itertools.combinations(range(k),2):
    n01=int(((F[:,a]==0)&(F[:,b]==1)).sum()); n10=int(((F[:,a]==1)&(F[:,b]==0)).sum())
    pr.append(dict(a=names[a],b=names[b],n01=n01,n10=n10,p=mcnemar([[0,n01],[n10,0]],exact=False,correction=True).pvalue))
PR=pd.DataFrame(pr); PR['p_holm']=multipletests(PR.p,method='holm')[1]; PR['sig']=PR.p_holm<0.05; print(PR.round(4)); PR.to_csv(f'{OUT}/pairwise_mcnemar_holm.csv',index=False)

## 5. Family, severity (within family) and position (within family) effects

In [ ]:
def wilson(k,n,z=1.96):
    p=k/n; den=1+z*z/n; c=(p+z*z/(2*n))/den; h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/den; return c-h,c+h
L=pd.concat([pd.read_parquet(f'{OUT}/long_{n}.parquet').assign(model=n) for n in names])
fam=L.groupby('family').agg(FR=('flip','mean'),pairs=('flip','size')); fam['pairs']=(fam.pairs/len(names)).astype(int)
fam['FRc']=L[L.ok_o==1].groupby('family').flip.mean(); fam['FRw']=L[L.ok_o==0].groupby('family').flip.mean()
fam['small(<1500)']=fam.pairs<1500; print(fam.sort_values('FR',ascending=False).round(3)); fam.to_csv(f'{OUT}/Table3_family.csv')
# severity within family: Page's L per (family, model) over reviews that have all 3 severities, Holm across tests
def page_L(M):                         # M: n x 3 (mild, medium, strong) FR per review
    r=np.apply_along_axis(stats.rankdata,1,M); n,k=M.shape; Lstat=(r*np.arange(1,k+1)).sum()
    mu=n*k*(k+1)**2/4; sd=math.sqrt(n*k*k*(k+1)*(k*k-1)/144); z=(Lstat-mu)/sd; return z,stats.norm.sf(z)
res=[]
for (fa,mo),g in L.groupby(['family','model']):
    pv=g.pivot_table(index='key',columns='severity',values='flip',aggfunc='mean')
    pv=pv[[c for c in ['mild','medium','strong'] if c in pv.columns]].dropna()
    if pv.shape[1]<3 or len(pv)<30: continue
    z,p=page_L(pv.values); res.append(dict(family=fa,model=mo,n_reviews=len(pv),FR_mild=pv.mild.mean(),FR_medium=pv.medium.mean(),FR_strong=pv.strong.mean(),z=z,p=p))
PL=pd.DataFrame(res); PL['p_holm']=multipletests(PL.p,method='holm')[1]; PL['p_bh']=multipletests(PL.p,method='fdr_bh')[1]
print(len(PL),'testable; Holm-significant:',int((PL.p_holm<.05).sum()),'BH:',int((PL.p_bh<.05).sum())); PL.round(4).to_csv(f'{OUT}/severity_pageL.csv',index=False)
# position within family (only families with >=2 positions)
pos=L.groupby(['family','position']).agg(FR=('flip','mean'),pairs=('flip','size')).reset_index(); pos['pairs']=(pos.pairs/len(names)).astype(int)
multi=pos.groupby('family').position.nunique(); print(pos[pos.family.isin(multi[multi>1].index)].round(3)); pos.to_csv(f'{OUT}/Table4_position_within_family.csv',index=False)
# DVI and its relation to dataset accuracy
dvi=L.groupby(['dataset','model']).flip.mean().groupby('dataset').mean().rename('DVI').to_frame().join(A3[['acc_o']]); print(dvi.corr().round(3)); dvi.to_csv(f'{OUT}/dvi.csv')

## 6. Fidelity: NLI-based meaning score instead of the degenerate Φ (+ VADER sanity check)

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vs=SentimentIntensityAnalyzer(); print('VADER on a clear example:',vs.polarity_scores('I love this, it is great')['compound'])   # must be != 0
u=df.drop_duplicates(['key','ptxt']).reset_index(drop=True) if False else df
emb=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=dev)
eo=emb.encode(df.orig.map(clean).tolist(),batch_size=256,normalize_embeddings=True,show_progress_bar=True); ep=emb.encode(df.ptxt.tolist(),batch_size=256,normalize_embeddings=True,show_progress_bar=True)
df['m1']=(eo*ep).sum(1)
df['vader_o']=df.orig.map(lambda t:vs.polarity_scores(clean(t))['compound']); df['vader_p']=df.ptxt.map(lambda t:vs.polarity_scores(t)['compound'])
assert df.vader_o.abs().sum()>0, 'VADER returns zeros -> fix before using M4'
df['m4']=(np.sign(df.vader_o)==np.sign(df.vader_p)).astype(int)
nli=CrossEncoder('cross-encoder/nli-deberta-v3-base',device=dev)      # labels: contradiction, entailment, neutral
def ent(a,b):
    s=nli.predict(list(zip(a,b)),batch_size=64,show_progress_bar=True,apply_softmax=True); return s[:,1]      # P(entailment)
# NLI on a subsample first (full corpus is large); extend as time permits
sub=df.sample(min(len(df),100000),random_state=SEED).copy()
sub['e_o2p']=ent(sub.orig.map(clean).tolist(),sub.ptxt.tolist()); sub['e_p2o']=ent(sub.ptxt.tolist(),sub.orig.map(clean).tolist())
sub['nli_min']=sub[['e_o2p','e_p2o']].min(1); sub['bidir_ok']=(sub.nli_min>0.5).astype(int)
print('share bidirectional-entailment:',sub.bidir_ok.mean().round(3)); print(sub.groupby('family').bidir_ok.mean().round(3))
sub[['key','family','severity','m1','m4','e_o2p','e_p2o','nli_min','bidir_ok']].to_csv(f'{OUT}/fidelity_subsample.csv',index=False)

## 7. Robustness restricted to pairs that are verified (bidirectional entailment) — the analysis reviewers will ask for

In [ ]:
fl=pd.concat([pd.read_parquet(f'{OUT}/long_{n}.parquet').assign(model=n) for n in names])
ok_keys=sub[sub.bidir_ok==1][['key','family','severity']].drop_duplicates()
rows=[]
for n in names:
    d=pd.read_parquet(f'{OUT}/long_{n}.parquet'); d=d.reset_index().rename(columns={'index':'row'}); d['k2']=d.key+'|'+d.family+'|'+d.severity
    okset=set((ok_keys.key+'|'+ok_keys.family+'|'+ok_keys.severity)); dd=d[d.k2.isin(okset)]
    rows.append(dict(model=n,RS_all=d.flip.mean(),RS_verified=dd.flip.mean(),n_verified=len(dd)))
print(pd.DataFrame(rows).round(4))

## 8. Second annotator → inter-annotator agreement and metric validity
Give `annotation_sheet_for_second_annotator.csv` and the guidelines to a second person (blind to the first labels). Upload the completed sheet and run the cell below.

In [ ]:
from sklearn.metrics import cohen_kappa_score, roc_auc_score
H=pd.read_csv('/content/annotated_output.csv')              # first annotator's file
S2=pd.read_csv('/content/annotation_sheet_completed.csv')   # second annotator's file (same sample_id)
M=H.merge(S2[['sample_id','meaning_a2','sentiment_a2','natural_a2']],on='sample_id')
for a,b in [('meaning_preserving','meaning_a2'),('sentiment_preserving','sentiment_a2'),('natural','natural_a2')]:
    cats=sorted(set(M[a])|set(M[b])); order={c:i for i,c in enumerate(['no','unclear','borderline','partial','yes'])}
    print(a,'weighted kappa =',round(cohen_kappa_score(M[a].map(order),M[b].map(order),weights='linear'),3),' exact agreement =',round((M[a]==M[b]).mean(),3))
# metric validity against consensus (both annotators say 'yes')
M['cons_yes']=((M.meaning_preserving=='yes')&(M.meaning_a2=='yes')).astype(int)
for c in ['m1_cosine_sim','m2_rougeL','m7_lsd','m8_pqs','sem_preservation_score']: print(c,'AUC vs consensus-preserving =',round(roc_auc_score(M.cons_yes,M[c]),3))

## 9. Export numbers for the manuscript

In [ ]:
nums={'Table2':T2.to_dict('records'),'cluster_CI':CI.to_dict('records'),'family':fam.reset_index().to_dict('records'),
      'pageL_testable':int(len(PL)),'pageL_holm_sig':int((PL.p_holm<.05).sum()),'dataset_audit':A3.reset_index().to_dict('records'),
      'mcnemar_holm':PR.round(4).to_dict('records')}
json.dump(nums,open(f'{OUT}/numbers.json','w'),indent=1,default=float)
# small zip: only the tables (a few KB) -- predictions/parquet and the 100k-row fidelity file are NOT included
import zipfile
with zipfile.ZipFile('cspf_tables.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT):
        p=f'{OUT}/{f}'
        if f.endswith(('.csv','.json')) and os.path.getsize(p)<2_000_000 and 'fidelity_subsample' not in f: z.write(p,f)
print('cspf_tables.zip size KB:',os.path.getsize('cspf_tables.zip')//1024)
# Even smaller: print the key tables as text; copy-paste this output into the chat if you cannot upload the zip
print(T2.to_string()); print(CI.to_string()); print(fam.round(3).to_string()); print(A3.to_string())
print('Page L testable',len(PL),'Holm sig',int((PL.p_holm<.05).sum()))